# VortexKit Aggregation Demo

This notebook demonstrates how to use the `aggregation` module to:

1. **Load** raw trade data from CSV into polars DataFrames
2. **Aggregate** trades → aggTrades (compressed trades)
3. **Aggregate** trades → klines (OHLCV candles) at **any** time interval

In [1]:
import sys
sys.path.insert(0, "..")

import polars as pl
from aggregation import *

## 1. Load Raw Trades

The loader reads a headerless Binance CSV, assigns canonical column names,
casts dtypes, and auto-detects timestamp precision (microseconds vs milliseconds).

In [2]:
DATA_DIR = "dataset"
SYMBOL = "BTCUSDT"
DATE = "2026-04-10"

trades = load_trades(f"{DATA_DIR}/{SYMBOL}-trades-{DATE}.csv")
print(f"Trades: {trades.shape}")
trades.head(10)

Trades: (3176308, 7)


trade_id,price,qty,quote_qty,time,is_buyer_maker,is_best_match
i64,f64,f64,f64,i64,bool,bool
6203596119,71787.98,0.00013,9.3324374,1775779200179713,false,true
6203596120,71787.98,0.00013,9.3324374,1775779200183132,false,true
6203596121,71787.98,0.00006,4.3072788,1775779200208187,false,true
6203596122,71787.98,0.00001,0.7178798,1775779200236198,false,true
6203596123,71787.98,0.00015,10.768197,1775779200236198,false,true
6203596124,71787.98,0.00012,8.6145576,1775779200236198,false,true
6203596125,71787.97,0.00017,12.203955,1775779200275423,true,true
6203596126,71787.98,0.00015,10.768197,1775779200434582,false,true
6203596127,71787.98,0.00015,10.768197,1775779200434582,false,true


In [3]:
# Verify timestamp precision
precision = detect_timestamp_precision(trades["time"])
print(f"Timestamp precision: {precision.value}")

Timestamp precision: us


## 2. Aggregate Trades → AggTrades

Groups consecutive trades with the same `(price, timestamp, is_buyer_maker)`.

In [4]:
agg = aggregate_trades(trades, start_agg_id=0)
print(f"AggTrades: {agg.shape}")
agg.head(10)

AggTrades: (910209, 8)


agg_trade_id,price,quantity,first_trade_id,last_trade_id,timestamp,was_buyer_maker,was_best_match
i64,f64,f64,i64,i64,i64,bool,bool
0,71787.98,0.00013,6203596119,6203596119,1775779200179713,false,true
1,71787.98,0.00013,6203596120,6203596120,1775779200183132,false,true
2,71787.98,0.00006,6203596121,6203596121,1775779200208187,false,true
3,71787.98,0.00028,6203596122,6203596124,1775779200236198,false,true
4,71787.97,0.00017,6203596125,6203596125,1775779200275423,true,true
5,71787.98,0.0018,6203596126,6203596129,1775779200434582,false,true
6,71787.98,0.01854,6203596130,6203596134,1775779200449703,false,true
7,71787.98,0.00086,6203596135,6203596135,1775779200531618,false,true
8,71787.98,0.00027,6203596136,6203596136,1775779200549679,false,true


## 3. Aggregate Trades → Klines (Any Interval)

The module supports **any** interval of the form `"<number><unit>"`:

| Unit | Meaning   | Example  |
|------|-----------|----------|
| `m`  | minutes   | `"3m"`   |
| `h`  | hours     | `"4h"`   |
| `d`  | days      | `"1d"`   |
| `w`  | weeks     | `"1w"`   |

In [5]:
# Standard intervals
klines_5m = aggregate_klines(trades, interval="5m")
klines_1h = aggregate_klines(trades, interval="1h")

print(f"5m klines: {klines_5m.shape}")
print(f"1h klines: {klines_1h.shape}")

5m klines: (288, 12)
1h klines: (24, 12)


In [6]:
# Custom intervals — any duration works
klines_3m = aggregate_klines(trades, interval="3m")
klines_4h = aggregate_klines(trades, interval="4h")
klines_4h = add_datetime_column(klines_4h, time_col="open_time", timezone="EST", new_col_name="open_time_est")
klines_4h = add_datetime_column(klines_4h, time_col="close_time", timezone="EST", new_col_name="close_time_est")
klines_6h = aggregate_klines(trades, interval="6h")
klines_2d = aggregate_klines(trades, interval="2d")

print(f"3m klines: {klines_3m.shape}")
print(f"4h klines: {klines_4h.shape}")
print(f"6h klines: {klines_6h.shape}")
print(f"2d klines: {klines_2d.shape}")

3m klines: (480, 12)
4h klines: (6, 14)
6h klines: (4, 12)
2d klines: (1, 12)


In [ ]:
# Inspect 4h klines
klines_4h

## 4. Verify Against Reference Data

Compare our 5m and 1h klines against Binance's official files.

In [ ]:
ref_5m = load_klines(f"{DATA_DIR}/{SYMBOL}-5m-{DATE}.csv")
ref_1h = load_klines(f"{DATA_DIR}/{SYMBOL}-1h-{DATE}.csv")

# Compare 5m
compare_5m = klines_5m.join(ref_5m, on="open_time", how="inner", suffix="_ref")
mismatches_5m = compare_5m.filter(
    pl.any_horizontal([
        (pl.col(c) != pl.col(f"{c}_ref"))
        for c in ["open","high","low","close","volume","quote_volume",
                   "num_trades","taker_buy_base_volume","taker_buy_quote_volume","close_time"]
    ])
)
print(f"5m klines: {klines_5m.shape[0]} rows, mismatches = {mismatches_5m.shape[0]}")

# Compare 1h
compare_1h = klines_1h.join(ref_1h, on="open_time", how="inner", suffix="_ref")
mismatches_1h = compare_1h.filter(
    pl.any_horizontal([
        (pl.col(c) != pl.col(f"{c}_ref"))
        for c in ["open","high","low","close","volume","quote_volume",
                   "num_trades","taker_buy_base_volume","taker_buy_quote_volume","close_time"]
    ])
)
print(f"1h klines: {klines_1h.shape[0]} rows, mismatches = {mismatches_1h.shape[0]}")

## 5. Interval Utility

The `interval_to_microseconds` function converts any interval string to microseconds.

In [ ]:
for iv in ["1m", "3m", "5m", "15m", "30m", "1h", "4h", "6h", "1d", "2d", "1w"]:
    us = interval_to_microseconds(iv)
    print(f"{iv:>4s} = {us:>15,} μs = {us / 1_000_000:>10,.0f} s")